In [1]:
!git clone -b claude/implement-giga-attack-011CUoJQFy9pvLvtQpC5uD7D https://github.com/mahopman/adc_llm_attack.git
%cd adc_llm_attack

Cloning into 'adc_llm_attack'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 160 (delta 85), reused 115 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 1.47 MiB | 22.40 MiB/s, done.
Resolving deltas: 100% (85/85), done.
/content/adc_llm_attack


In [2]:
!uv sync

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 150 packages in 10ms
Prepared 77 packages in 1m 22s                                           
Installed 77 packages in 597ms                              
 + accelerate==1.11.0
 + adc-llm-attack==0.1.0 (from file:///content/adc_llm_attack)
 + asttokens==3.0.0
 + certifi==2025.10.5
 + charset-normalizer==3.4.4
 + comm==0.2.3
 + debugpy==1.8.17
 + decorator==5.2.1
 + executing==2.2.1
 + filelock==3.20.0
 + fsspec==2025.10.0
 + hf-transfer==0.1.9
 + hf-xet==1.2.0
 + huggingface-hub==0.36.0
 + idna==3.11
 + ipykernel==7.1.0
 + ipython==9.7.0
 + ipython-pygments-lexers==1.1.1
 + ipywidgets==8.1.8
 + jedi==0.19.2
 + jinja2==3.1.6
 + jupyter-client==8.6.3
 + jupyter-core==5.9.1
 + jupyterlab-widgets==3.0.16
 + markupsafe==3.0.3
 + matplotlib-inline==0.2.1
 + mpmath==1.3.0
 + nest-asyncio==1.6.0
 + networkx==3.5
 + numpy==2.3.4
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.

In [3]:
import torch
import numpy as np
import time
from llm_attack import GIGAAttack
from utils import get_input_template, get_model

# Set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)

In [4]:
model_name = 'lmsys/vicuna-7b-v1.5'

print(f"\nLoading model: {model_name}")
model, tokenizer = get_model(model_name)
model.eval()


Loading model: lmsys/vicuna-7b-v1.5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,),

In [5]:
user_prompt = "What is the capital of the USA?"

# Vicuna format (manual)
input_text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions.\n\nUSER: {user_prompt}\nASSISTANT:"

# Tokenize
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)

# Generate response
print("Generating target response...")
with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=100,
        do_sample=False,  # Greedy decoding for consistency
        pad_token_id=tokenizer.pad_token_id
    )

# Decode only the generated part (exclude input)
generated_ids = output_ids[0][input_ids.shape[1]:]
target_response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"\nUser prompt: {user_prompt}")
print(f"Target response: {target_response}")

Generating target response...

User prompt: What is the capital of the USA?
Target response: The capital of the USA is Washington, D.C.


In [ ]:
NUM_ADV_TOKENS = 5

In [24]:
adv_placeholder = " !" * NUM_ADV_TOKENS

def build_input_messages(adv_suffix, user_prompt):
    return [
        {"role": "system", "content": "A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions."},
        {"role": "user", "content": f"{user_prompt} {adv_suffix}"},
        {"role": "assistant", "content": target_response}
    ]

starting_text = build_input_messages(adv_placeholder, user_prompt)


### Step 1: find adv. suffix that repeats (ignore the target response)

In [25]:

from utils.llm_utils import get_chat_template
tokenizer.chat_template = get_chat_template('vicuna')

full_string = tokenizer.apply_chat_template(
    starting_text,
    tokenize=False,
    add_generation_prompt=False  # We include assistant response
)

input_ids = tokenizer.encode(full_string, add_special_tokens=True)

print(f"  Total tokens: {len(input_ids)}")
print(f"\n  Full template:\n{full_string}")


  Total tokens: 69

  Full template:
<s>A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions.

USER: What is the capital of the USA?  ! ! ! ! !
ASSISTANT: The capital of the USA is Washington, D.C.</s>



In [44]:
adv_input_start = None
adv_input_stop = None
response_start = None
response_stop = len(input_ids)

total_len = len(input_ids)

for i in range(total_len, 0, -1):
    decoded = tokenizer.decode(input_ids[i:])

    if response_start is None:
        if target_response in decoded or target_response.strip() in decoded:
            response_start = i
            break

for i in range(response_start, 0, -1):
    decoded = tokenizer.decode(input_ids[i:])

    if adv_input_start is None:
        if adv_placeholder.strip() in decoded or adv_placeholder in decoded:
            adv_input_start = i
            adv_input_stop = i + NUM_ADV_TOKENS
            break

input_slice = slice(0, adv_input_start)
adv_slice = slice(adv_input_start, adv_input_stop)
response_slice = slice(response_start, response_stop)

print(f"Input slice: {input_slice} ({adv_input_start} tokens)")
print(f"Input ADV slice: {adv_slice} ({adv_input_stop - adv_input_start} tokens)")
print(f"Response slice: {response_slice} ({response_stop - response_start} tokens)")

print(f"\n  Verification:")
print(f"Input: '{tokenizer.decode(input_ids[input_slice])}'")
print(f"Input ADV: '{tokenizer.decode(input_ids[adv_slice])}'")
print(f"Response: '{tokenizer.decode(input_ids[response_slice])}'")

Input slice: slice(0, 45, None) (45 tokens)
Input ADV slice: slice(45, 50, None) (5 tokens)
Response slice: slice(55, 69, None) (14 tokens)

  Verification:
Input: '<s><s>A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions.

USER: What is the capital of the USA? '
Input ADV: '! ! ! ! !'
Response: 'The capital of the USA is Washington, D.C.</s>
'


In [46]:
embed_layer = model.model.embed_tokens
embedding_matrix = embed_layer.weight
vocab_size = embedding_matrix.shape[0]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
dtype = torch.float16

In [51]:
input_ids_tensor = torch.tensor(input_ids, device=device)

# Pre-compute embeddings for parts that don't change
embed_before_input_adv = embed_layer(input_ids_tensor[:adv_input_start])
embed_between = embed_layer(input_ids_tensor[adv_input_stop:response_start])
embed_response = embed_layer(input_ids_tensor[response_slice])


In [40]:
import torch.nn.functional as F

In [48]:
soft_adv = torch.randn(NUM_ADV_TOKENS, vocab_size, device=device)
soft_adv = F.softmax(soft_adv, dim=-1)
soft_adv.requires_grad = True

print(f"  Soft ADV shape: {soft_adv.shape}")
print(f"  Sum of probabilities: {soft_adv[0].sum():.4f} (should be ~1.0)")

  Soft ADV shape: torch.Size([5, 32000])
  Sum of probabilities: 1.0000 (should be ~1.0)


In [49]:
learning_rate = 0.01
num_steps = 5000

optimizer = torch.optim.Adam([soft_adv], lr=learning_rate)

In [53]:
best_loss = float('inf')
best_adv_tokens = None

for step in range(num_steps):
    optimizer.zero_grad()

    # Task 11: Get current discrete adversarial tokens
    current_adv_tokens = soft_adv.argmax(dim=-1)

    # Get soft and hard embeddings
    adv_embeds_soft = soft_adv @ embedding_matrix.float()  # Soft (for input)
    adv_embeds_hard = embed_layer(current_adv_tokens)       # Hard (for output)
    
    # Task 11: Build full embeddings
    # Input ADV uses SOFT embeddings (gradient flows)
    # Output ADV uses HARD embeddings (teacher forcing)
    full_embeds = torch.cat([
        embed_before_input_adv,  # Before input ADV
        adv_embeds_soft,         # Input ADV (SOFT)
        embed_between,           # Between input ADV and response
        embed_response,          # Response: "Washington DC"
        adv_embeds_hard          # Output ADV (HARD - current discrete)
    ], dim=0).unsqueeze(0)

    # Task 12: Forward pass
    outputs = model(inputs_embeds=full_embeds.to(dtype))
    logits = outputs.logits

    # Task 13: Extract target logits
    # We want to predict: [current_adv_tokens]
    # The logits are shifted by 1 for next-token prediction
    target_tokens = current_adv_tokens

    # Get logits for positions we care about (shifted by -1)
    target_logits = logits[0, -(NUM_ADV_TOKENS + 1):-1]

    # Task 14: Compute loss
    loss = F.cross_entropy(target_logits, target_tokens)

    # Task 15: Backpropagate
    loss.backward()
    optimizer.step()

    # Re-normalize to maintain valid probability distribution
    with torch.no_grad():
        soft_adv.data = F.softmax(soft_adv.data, dim=-1)

    # Track best
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_adv_tokens = current_adv_tokens.clone()

    # Print progress
    if (step + 1) % 50 == 0 or step == 0:
        adv_string = tokenizer.decode(current_adv_tokens)
        print(f"  Step {step+1:3d}/{num_steps} | Loss: {loss.item():.4f} | ADV: '{adv_string[:30]}...'")

print(f"\n   Optimization complete!")
print(f"  Best loss: {best_loss:.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 9359 has 14.74 GiB memory in use. Of the allocated memory 14.59 GiB is allocated by PyTorch, and 24.28 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)